[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/18_montecarlo_search/notebooks/02_mcts_paso_a_paso.ipynb)

# Notebook 2: MCTS paso a paso

**Módulo 18 — Monte Carlo Tree Search · Notebook 02**

---

## ¿De qué trata este notebook?

En este notebook implementamos **MCTS vanilla** (con selección greedy por $Q/N$)
y lo ejecutamos paso a paso para entender las cuatro fases del algoritmo.

**Lo que vas a construir y explorar:**

1. La clase `MCTSNode` y el algoritmo MCTS completo.
2. Traza manual: 20 iteraciones impresas con estadísticas.
3. Visualización de las visitas a los hijos de la raíz.
4. Ejercicio: comparar MCTS con distintos presupuestos de iteraciones.

**Relación con las notas de clase:** sección 03 (MCTS: las cuatro fases).

**Prerequisitos:** Notebook 01 (clase `Hex` y rollouts).

---

In [ ]:
# Solo en Colab
# !pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import copy
from collections import deque

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB",
    "red": "#E94F37",
    "green": "#27AE60",
    "gray": "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
}

random.seed(42)
np.random.seed(42)

---

## 1. Clase `Hex` (reimplementación)

Para que este notebook sea autocontenido, reimplementamos la clase `Hex`.

In [ ]:
class Hex:
    """Juego de Hex en un tablero de size x size."""

    NEIGHBORS = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]

    def __init__(self, size=7):
        self.size = size
        self.board = [[0] * size for _ in range(size)]
        self.current_player = 1

    def actions(self):
        """Retorna lista de celdas vacías (r, c)."""
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.board[r][c] == 0]

    def result(self, action):
        """Aplica la acción y retorna un nuevo estado."""
        new = copy.deepcopy(self)
        r, c = action
        new.board[r][c] = self.current_player
        new.current_player = 3 - self.current_player
        return new

    def is_terminal(self):
        return self._has_path(1) or self._has_path(2)

    def utility(self, player):
        if self._has_path(player):
            return 1
        if self._has_path(3 - player):
            return -1
        return 0

    def _has_path(self, player):
        """BFS para verificar conexión entre lados."""
        n = self.size
        if player == 1:
            start = [(0, c) for c in range(n) if self.board[0][c] == 1]
            goal = lambda r, c: r == n - 1
        else:
            start = [(r, 0) for r in range(n) if self.board[r][0] == 2]
            goal = lambda r, c: c == n - 1
        visited = set()
        queue = deque(start)
        while queue:
            r, c = queue.popleft()
            if (r, c) in visited:
                continue
            visited.add((r, c))
            if goal(r, c):
                return True
            for dr, dc in self.NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and self.board[nr][nc] == player:
                    queue.append((nr, nc))
        return False

---

## 2. Clase `MCTSNode`

Cada nodo del árbol almacena:
- `state`: estado del juego
- `parent`: nodo padre
- `children`: diccionario `{acción: nodo_hijo}`
- `N`: número de visitas
- `Q`: valor acumulado (suma de resultados)
- `unexpanded`: acciones legales sin nodo hijo

In [ ]:
class MCTSNode:
    """Nodo del árbol de MCTS.

    Almacena estadísticas de visitas (N) y valor acumulado (Q)
    para guiar la búsqueda.
    """

    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}           # {acción: MCTSNode}
        self.N = 0                   # número de visitas
        self.Q = 0.0                 # valor acumulado
        # Acciones legales que todavía no tienen nodo hijo
        self.unexpanded = list(state.actions()) if not state.is_terminal() else []

    def win_rate(self):
        """Tasa de éxito Q/N."""
        if self.N == 0:
            return 0.0
        return self.Q / self.N

    def __repr__(self):
        return f"MCTSNode(N={self.N}, Q={self.Q:.1f}, Q/N={self.win_rate():.3f})"

---

## 3. Algoritmo MCTS vanilla

Implementamos MCTS con la política de selección más simple: elegir el hijo
con mayor $Q/N$. Es **pura explotación** — en el notebook 03 la reemplazaremos
por UCT.

Las cuatro fases del pseudocódigo:
- `[M1]` Selección: bajar por el árbol eligiendo el mejor hijo
- `[M2]` Expansión: añadir un nodo nuevo
- `[M3]` Simulación: rollout aleatorio
- `[M4]` Retropropagación: actualizar N y Q hacia arriba

In [ ]:
def mcts(game_state, iterations, player, verbose=False):
    """MCTS vanilla con selección greedy (mayor Q/N).

    Parameters
    ----------
    game_state : Hex
        Estado actual del juego.
    iterations : int
        Presupuesto de iteraciones.
    player : int
        Jugador desde cuya perspectiva medimos utilidad (1 o 2).
    verbose : bool
        Si True, imprime información de cada iteración.

    Returns
    -------
    best_action : tuple
        La acción más visitada en la raíz.
    root : MCTSNode
        El nodo raíz con todas las estadísticas.
    """
    root = MCTSNode(game_state)

    for i in range(iterations):
        node = root

        # ── [M1] Selección ─────────────────────────────────────────────
        # Bajar por el árbol eligiendo el hijo con mayor Q/N
        while not node.unexpanded and node.children:
            node = max(node.children.values(), key=lambda c: c.Q / c.N)

        # ── [M2] Expansión ─────────────────────────────────────────────
        # Añadir un nodo hijo para una acción no explorada
        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        # ── [M3] Simulación ────────────────────────────────────────────
        # Rollout aleatorio desde el nodo actual
        sim = copy.deepcopy(node.state)
        while not sim.is_terminal():
            sim = sim.result(random.choice(sim.actions()))
        reward = sim.utility(player)

        # ── [M4] Retropropagación ──────────────────────────────────────
        # Actualizar N y Q desde el nodo actual hasta la raíz
        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

        # Información de traza
        if verbose and (i < 20 or (i + 1) % 50 == 0):
            stats = []
            for a, ch in sorted(root.children.items()):
                stats.append(f"{a}:N={ch.N},Q/N={ch.win_rate():.2f}")
            print(f"Iter {i+1:3d} | reward={reward:+d} | hijos: {', '.join(stats[:5])}{'...' if len(stats) > 5 else ''}")

    # Elegir la acción más visitada
    best_action = max(root.children, key=lambda a: root.children[a].N)
    return best_action, root

---

## 4. Traza manual: 20 iteraciones en Hex 3×3

Ejecutemos MCTS en el tablero vacío de 3×3 con modo `verbose` para ver
exactamente qué ocurre en cada iteración.

In [ ]:
random.seed(42)

state = Hex(size=3)
print("Estado: tablero vacío de Hex 3×3, turno de Negro\n")
print("Primeras 20 iteraciones de MCTS vanilla:")
print("=" * 80)

best_action, root = mcts(state, iterations=20, player=1, verbose=True)

print("=" * 80)
print(f"\nAcción elegida (más visitada): {best_action}")
print(f"\nEstadísticas de la raíz: N={root.N}, Q={root.Q:.0f}, Q/N={root.win_rate():.3f}")
print(f"\nEstadísticas de cada hijo:")
for action, child in sorted(root.children.items()):
    print(f"  Celda {action}: N={child.N:3d}, Q={child.Q:+5.0f}, Q/N={child.win_rate():+.3f}")

### Observaciones de la traza

1. **Iteraciones 1-9**: se expande un hijo nuevo por iteración (las 9 celdas).
   Cada hijo tiene $N=1$ al final.
2. **Iteración 10+**: `[M1]` empieza a funcionar — selecciona el hijo con
   mayor $Q/N$ y expande su subárbol.
3. **Problema**: la selección greedy puede quedarse atrapada en el primer
   hijo que tuvo un buen rollout, ignorando alternativas.

---

## 5. Más iteraciones y visualización

Ejecutemos MCTS con más iteraciones y visualicemos cómo se reparten las
visitas entre los hijos de la raíz.

In [ ]:
random.seed(42)

state = Hex(size=3)
best_action, root = mcts(state, iterations=500, player=1)

# Extraer datos de los hijos de la raíz
acciones = sorted(root.children.keys())
visitas = [root.children[a].N for a in acciones]
win_rates = [root.children[a].win_rate() for a in acciones]
etiquetas = [f"({r},{c})" for r, c in acciones]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Visitas por acción
bars = ax1.bar(etiquetas, visitas, color=COLORS["blue"], alpha=0.8)
# Resaltar la más visitada
max_idx = np.argmax(visitas)
bars[max_idx].set_color(COLORS["green"])
ax1.set_xlabel("Celda (fila, columna)")
ax1.set_ylabel("Visitas (N)")
ax1.set_title("Visitas por acción — MCTS vanilla (500 iter)")
ax1.tick_params(axis='x', rotation=45)

# Gráfica 2: Tasa de éxito por acción
bars2 = ax2.bar(etiquetas, win_rates, color=COLORS["orange"], alpha=0.8)
bars2[max_idx].set_color(COLORS["green"])
ax2.set_xlabel("Celda (fila, columna)")
ax2.set_ylabel("Tasa de éxito (Q/N)")
ax2.set_title("Tasa de éxito por acción — MCTS vanilla (500 iter)")
ax2.axhline(0, color=COLORS["gray"], linestyle="--", alpha=0.5)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Acción elegida (más visitada): {best_action}")
print(f"Visitas de la raíz: {root.N}")

### Nota sobre la selección greedy

Observa que las visitas están muy **concentradas**: una o dos acciones acaparan
la mayoría. Esto es el problema de la pura explotación — la política $Q/N$
se aferra a lo primero que funciona.

---

## 6. Crecimiento del árbol

¿Cuántos nodos tiene el árbol después de $M$ iteraciones? Tracemos el
crecimiento del número de nodos conforme avanza la búsqueda.

In [ ]:
def contar_nodos(node):
    """Cuenta el número total de nodos en el subárbol."""
    total = 1
    for child in node.children.values():
        total += contar_nodos(child)
    return total


def mcts_con_historial(game_state, iterations, player):
    """MCTS que registra el número de nodos después de cada iteración."""
    root = MCTSNode(game_state)
    historial_nodos = []

    for i in range(iterations):
        node = root

        # [M1] Selección
        while not node.unexpanded and node.children:
            node = max(node.children.values(), key=lambda c: c.Q / c.N)

        # [M2] Expansión
        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        # [M3] Simulación
        sim = copy.deepcopy(node.state)
        while not sim.is_terminal():
            sim = sim.result(random.choice(sim.actions()))
        reward = sim.utility(player)

        # [M4] Retropropagación
        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

        # Registrar nodos (solo cada 10 iteraciones para eficiencia)
        if (i + 1) % 10 == 0 or i < 10:
            historial_nodos.append((i + 1, contar_nodos(root)))

    return root, historial_nodos


random.seed(42)
state = Hex(size=3)
root, historial = mcts_con_historial(state, iterations=500, player=1)

iters, nodos = zip(*historial)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iters, nodos, color=COLORS["purple"], linewidth=2, marker='o', markersize=3)
ax.set_xlabel("Iteraciones")
ax.set_ylabel("Nodos en el árbol")
ax.set_title("Crecimiento del árbol MCTS — Hex 3×3")
ax.axhline(len(state.actions()), color=COLORS["gray"], linestyle="--", alpha=0.5,
           label=f"Acciones legales = {len(state.actions())}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Nodos finales: {nodos[-1]}")
print(f"Nota: el árbol completo de Hex 3×3 tiene ~10^3 nodos.")
print(f"MCTS solo expande una fracción del árbol total.")

---

## 7. Ejercicio: comparar presupuestos de iteraciones

**Tu tarea:** ejecuta MCTS con distintos presupuestos (50, 100, 200, 500)
y compara:
1. ¿Cambia la acción elegida con más iteraciones?
2. ¿Cómo se distribuyen las visitas en cada caso?
3. ¿La selección greedy concentra demasiado las visitas?

Completa el código y analiza los resultados.

In [ ]:
random.seed(42)

state = Hex(size=3)
presupuestos = [50, 100, 200, 500]

fig, axes = plt.subplots(1, len(presupuestos), figsize=(16, 4), sharey=True)

for ax, M in zip(axes, presupuestos):
    random.seed(42)  # misma semilla para comparar
    best, root = mcts(state, iterations=M, player=1)

    acciones = sorted(root.children.keys())
    visitas = [root.children[a].N for a in acciones]
    etiquetas = [f"({r},{c})" for r, c in acciones]

    bars = ax.bar(etiquetas, visitas, color=COLORS["blue"], alpha=0.8)
    # Resaltar la elegida
    for j, a in enumerate(acciones):
        if a == best:
            bars[j].set_color(COLORS["green"])

    ax.set_title(f"M = {M}\nEligió: {best}")
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    if ax == axes[0]:
        ax.set_ylabel("Visitas (N)")

plt.suptitle("Distribución de visitas — MCTS vanilla con distintos presupuestos",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n¿Qué observas sobre la concentración de visitas?")
print("En el notebook 03 veremos cómo UCT resuelve este problema.")

---

## 8. Limitaciones de la selección greedy

La selección por $Q/N$ (greedy / pura explotación) tiene un problema fundamental:

1. **Se aferra al primer éxito**: un hijo que ganó su primer rollout tiene $Q/N = 1.0$
   y recibe todas las visitas siguientes. Otros hijos con potencial quedan ignorados.

2. **No explora lo suficiente**: es exactamente el mismo error que cometer pura
   explotación en el problema de bandidos (módulo 17).

3. **La solución**: añadir un **término de exploración** — exactamente lo que hace
   UCB1. Aplicado a MCTS, se llama **UCT** (Upper Confidence bounds applied to Trees).

Esto es el tema del **notebook 03**.

---